In [1]:
#@title CELL 1 — Cài thư viện, Java 11 và kiểm tra bộ chấm điểm
"""
Baseline Level 1 • COCO 2014 Karpathy • seeds 42, 44, 46 • beam=3, alpha=0.6.
Chọn GPU; ở Cell 2 chọn riêng seed 42, 44 hoặc 46 rồi chạy lần lượt 6 cell.
Chạy một phiên cho mỗi seed; có thể chạy các seed khác nhau độc lập.
Đầu ra mới: baseline_level1/multiseed_beam3/run_<protocol>/seed_<seed>/.
Dữ liệu gốc và kết quả cũ không được sửa. File không khớp sẽ được giữ nguyên.
Predictions JSON gồm metadata và danh sách COCO rows tại khóa "predictions".
Ngắt giữa epoch: chạy lại epoch đó; Beam Search lưu tiến độ mỗi 100 ảnh.
Nếu runtime chết còn khóa: chỉ bật OLD_SESSION_STOPPED khi phiên cũ đã dừng hẳn.
Seed 42 dùng checkpoint cũ; seed 44/46 tối đa 15 epoch, patience=3.
MAX_LEN=30 tính cả BOS/EOS; kết quả dùng thang gốc của pycocoevalcap.
METEOR ở đây là Java METEOR của pycocoevalcap, khác NLTK trong báo cáo cũ.
Lưu ý: checkpoint seed 42 được tạo với budget cũ tối đa 30 epoch.
Đã kiểm tra checkpoint seed 42 và Beam Search trên dữ liệu thật bằng CPU;
chưa chạy train 44/46 hoặc chấm toàn bộ 5.000 ảnh trên GPU Colab.
Không thể bảo đảm Colab/Drive không mất kết nối hoặc hết tài nguyên.
"""
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import sys, subprocess, shutil, tempfile, tarfile, urllib.request, zipfile
from pathlib import Path
from importlib import metadata

RUNTIME_DIR = Path("/content/level1_beam3_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pycocoevalcap==1.2", "nltk>=3.9,<4", "tqdm>=4.66,<5"],
    check=True,
)

def java11_version(java):
    result = subprocess.run([str(java), "-version"], capture_output=True, text=True)
    return result.returncode == 0 and 'version "11.' in result.stderr

java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java"))
java_candidates += list((RUNTIME_DIR / "java11").glob("*/bin/java"))
if not any(java11_version(p) for p in java_candidates):
    try:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq",
                        "openjdk-11-jre-headless"], check=True)
    except subprocess.CalledProcessError:
        # Colab Ubuntu mới có thể không cung cấp package OpenJDK 11.
        java_root = RUNTIME_DIR / "java11"
        java_root.mkdir(exist_ok=True)
        with tempfile.TemporaryDirectory(dir=RUNTIME_DIR) as tmp:
            archive = Path(tmp) / "jdk11.tar.gz"
            urllib.request.urlretrieve(
                "https://api.adoptium.net/v3/binary/latest/11/ga/linux/x64/jre/hotspot/normal/eclipse",
                archive,
            )
            with tarfile.open(archive, "r:gz") as tar:
                tar.extractall(java_root, filter="data")
    java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java"))
    java_candidates += list((RUNTIME_DIR / "java11").glob("*/bin/java"))
JAVA_BIN = next((p for p in java_candidates if java11_version(p)), None)
if JAVA_BIN is None:
    raise RuntimeError("Không cài được Java 11; xem log cài đặt ở trên.")
os.environ["JAVA_HOME"] = str(JAVA_BIN.parent.parent)
os.environ["PATH"] = str(JAVA_BIN.parent) + os.pathsep + os.environ["PATH"]
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

import nltk
for resource in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
    if not nltk.download(resource, quiet=True):
        raise RuntimeError(f"Không tải được NLTK resource: {resource}")

import pycocoevalcap.spice.spice as spice_module
SPICE_DIR = Path(spice_module.__file__).parent
SPICE_LIB = SPICE_DIR / "lib"
SPICE_LIB.mkdir(exist_ok=True)
# Tải cả code jar và models jar qua HTTPS, tránh downloader HTTP cũ.
for suffix in ("", "-models"):
    jar_name = f"stanford-corenlp-3.6.0{suffix}.jar"
    jar_path = SPICE_LIB / jar_name
    if not jar_path.exists() or not zipfile.is_zipfile(jar_path):
        url = ("https://repo.maven.apache.org/maven2/edu/stanford/nlp/"
               f"stanford-corenlp/3.6.0/{jar_name}")
        with tempfile.TemporaryDirectory(dir=RUNTIME_DIR) as tmp:
            downloaded = Path(tmp) / jar_name
            urllib.request.urlretrieve(url, downloaded)
            if not zipfile.is_zipfile(downloaded):
                raise RuntimeError(f"Jar tải về không hợp lệ: {jar_name}")
            shutil.copyfile(downloaded, jar_path)

# Package và cache SPICE đều nằm trong runtime; không ghi công cụ lên Drive.
spice_module.TEMP_DIR = str(RUNTIME_DIR / "spice_tmp")
spice_module.CACHE_DIR = str(RUNTIME_DIR / "spice_cache")
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.spice.spice import Spice
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

class ManagedMeteor(Meteor):
    def close(self):
        process = getattr(self, "meteor_p", None)
        if process is not None and process.poll() is None:
            try:
                if process.stdin is not None:
                    process.stdin.close()
            except (OSError, ValueError):
                pass
            finally:
                process.kill()
                process.wait(timeout=30)
    def __del__(self):
        self.close()

def close_meteor(scorer):
    scorer.close()

_smoke_gts = PTBTokenizer().tokenize({1: [{"caption": "a cat sits on a mat"}]})
_smoke_res = PTBTokenizer().tokenize({1: [{"caption": "a cat sits on a mat"}]})
_meteor = ManagedMeteor()
try:
    _meteor_score, _ = _meteor.compute_score(_smoke_gts, _smoke_res)
finally:
    close_meteor(_meteor)
del _meteor
_spice_score, _ = Spice().compute_score(_smoke_gts, _smoke_res)
import math
if not all(math.isfinite(float(x)) for x in (_meteor_score, _spice_score)):
    raise RuntimeError("Smoke test METEOR/SPICE thất bại.")
print(f"Bộ chấm điểm OK • METEOR={_meteor_score:.4f} • SPICE={_spice_score:.4f}")
print("pycocoevalcap", metadata.version("pycocoevalcap"), "| NLTK", nltk.__version__)
# Nguồn triển khai: https://github.com/salaniz/pycocoevalcap


openjdk version "11.0.32.1" 2026-08-18
OpenJDK Runtime Environment (build 11.0.32.1+1-post-1ubuntu1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 11.0.32.1+1-post-1ubuntu1-24.04-Ubuntu, mixed mode, sharing)

Bộ chấm điểm OK • METEOR=1.0000 • SPICE=1.0000
pycocoevalcap 1.2 | NLTK 3.9.1


In [2]:
#@title CELL 2 — Chọn seed, Mount Drive và kiểm tra dữ liệu (chỉ đọc dữ liệu gốc)
# Chạy riêng 42, rồi đổi sang 44, rồi 46; mỗi lần chạy lại lần lượt 6 cell.
SEED_TO_RUN = "42" #@param ["42", "44", "46"]
# Chỉ bật nếu phiên Colab cũ cùng seed đã dừng hẳn nhưng còn khóa sau disconnect.
OLD_SESSION_STOPPED = False #@param {type:"boolean"}
SELECTED_SEED = int(SEED_TO_RUN)
# Giữ cùng tag cho 3 seed. Chỉ đổi tag khi muốn một thí nghiệm MỚI độc lập.
EXPERIMENT_TAG = "main" #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import json, re, random, hashlib, platform, time, gc
from collections import defaultdict
if not re.fullmatch(r"[A-Za-z0-9_-]{1,40}", EXPERIMENT_TAG):
    raise ValueError("EXPERIMENT_TAG chỉ gồm chữ không dấu, số, gạch dưới/gạch ngang, tối đa 40 ký tự.")
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import display, Markdown

DATA_ROOT = Path("/content/drive/MyDrive/DataKhoaLuan/Data")
BASE_DIR = DATA_ROOT / "baseline_level1"
FEATURE_DIR = BASE_DIR / "clip_image_features"
OUTPUT_ROOT = BASE_DIR / "multiseed_beam3"
SEED42_CHECKPOINT = BASE_DIR / "checkpoints/checkpoint_best.pt"
MAX_LEN, BATCH_SIZE = 30, 128
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Hãy bật GPU trong Colab trước khi train/đánh giá toàn bộ dữ liệu.")

def read_json(path):
    with Path(path).open(encoding="utf-8") as f:
        return json.load(f)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True,
        ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()

vocab_path = BASE_DIR / "vocab.json"
vocab = read_json(vocab_path)
stoi = {str(k): int(v) for k, v in vocab["stoi"].items()}
raw_itos = vocab["itos"]
itos = ([str(x) for x in raw_itos] if isinstance(raw_itos, list)
        else [str(raw_itos[str(i)]) for i in range(len(stoi))])
VOCAB_SIZE = len(stoi)
assert VOCAB_SIZE == 16214, f"Vocab hiện tại khác dữ liệu đã kiểm tra: {VOCAB_SIZE}"
assert sorted(stoi.values()) == list(range(VOCAB_SIZE))
assert all(stoi.get(t) == i for i, t in enumerate(("<PAD>", "<BOS>", "<EOS>", "<UNK>")))
assert all(itos[i] == t for t, i in stoi.items()), "stoi/itos không khớp."
assert int(vocab.get("max_length", MAX_LEN)) == MAX_LEN

# Đường dẫn thực tế đã kiểm tra trên Drive, kèm đường dẫn ngắn trong yêu cầu.
annotation_candidates = [
    DATA_ROOT / "COCO_2014/annotations_trainval2014/annotations",
    DATA_ROOT / "COCO_2014/annotations",
]
ANNOTATION_DIR = next((p for p in annotation_candidates if all(
    (p / f"captions_karpathy_{s}.json").is_file()
    for s in ("train", "val", "test"))), None)
if ANNOTATION_DIR is None:
    raise FileNotFoundError("Không tìm được đủ 3 annotations Karpathy tại: "
                            + ", ".join(map(str, annotation_candidates)))
if not SEED42_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Thiếu checkpoint seed 42: {SEED42_CHECKPOINT}")

TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
def encode_caption(text):
    words = TOKEN_PATTERN.findall(text.lower())[:MAX_LEN - 2]
    values = [BOS_ID] + [stoi.get(w, UNK_ID) for w in words] + [EOS_ID]
    encoded = np.full(MAX_LEN, PAD_ID, dtype=np.int32)
    encoded[:len(values)] = values
    return encoded

class CaptionDataset(Dataset):
    def __init__(self, features, image_ids, annotations):
        self.features = features
        row_by_id = {image_id: i for i, image_id in enumerate(image_ids)}
        grouped = defaultdict(list)
        for annotation in annotations:
            if str(annotation["caption"]).strip():
                grouped[int(annotation["image_id"])].append(annotation)
        # Giữ thứ tự image/caption của dataset baseline cũ trước khi shuffle.
        usable = [a for image_id in sorted(image_ids) for a in grouped[image_id]]
        self.feature_rows = np.asarray(
            [row_by_id[int(a["image_id"])] for a in usable], dtype=np.int32)
        self.tokens = np.stack([encode_caption(a["caption"]) for a in usable])
    def __len__(self):
        return len(self.feature_rows)
    def __getitem__(self, index):
        feature = torch.from_numpy(self.features[self.feature_rows[index]])
        caption = torch.from_numpy(self.tokens[index].astype(np.int64))
        return feature, caption

class ImageDataset(Dataset):
    def __init__(self, features, image_ids):
        self.features, self.image_ids = features, image_ids
    def __len__(self):
        return len(self.image_ids)
    def __getitem__(self, index):
        return self.image_ids[index], torch.from_numpy(self.features[index])

features, image_ids, caption_datasets = {}, {}, {}
references = None
DATA_FINGERPRINTS = {"vocab": sha256_file(vocab_path)}
expected_images = {"train": 113287, "val": 5000, "test": 5000}
expected_captions = {"train": 566747, "val": 25010, "test": 25010}
for split in ("train", "val", "test"):
    feature_path = FEATURE_DIR / f"{split}_features.npy"
    id_path = FEATURE_DIR / f"{split}_image_ids.json"
    annotation_path = ANNOTATION_DIR / f"captions_karpathy_{split}.json"
    for p in (feature_path, id_path, annotation_path):
        if not p.is_file():
            raise FileNotFoundError(p)
        DATA_FINGERPRINTS[f"{split}/{p.name}"] = sha256_file(p)
    # Nạp một lần vào RAM; không đọc ngẫu nhiên trên Drive trong từng minibatch.
    array = np.load(feature_path, allow_pickle=False)
    ids = [int(x) for x in read_json(id_path)]
    payload = read_json(annotation_path)
    annotations = payload["annotations"]
    assert array.shape == (expected_images[split], 512), (split, array.shape)
    assert array.dtype == np.float32 and np.isfinite(array).all(), split
    assert len(ids) == len(set(ids)) == len(array), f"IDs trùng/lệch: {split}"
    assert set(ids) == {int(a["id"]) for a in payload["images"]}, split
    refs = defaultdict(list)
    for a in annotations:
        if str(a["caption"]).strip():
            refs[int(a["image_id"])].append(str(a["caption"]))
    assert set(ids) == set(refs), f"Thiếu/thừa caption tham chiếu: {split}"
    assert sum(map(len, refs.values())) == expected_captions[split], split
    assert np.allclose(np.linalg.norm(array, axis=1), 1.0, atol=1e-3), (
        f"Cache {split} không còn là đặc trưng CLIP đã chuẩn hóa.")
    features[split], image_ids[split] = array, ids
    if split in ("train", "val"):
        caption_datasets[split] = CaptionDataset(array, ids, annotations)
    else:
        references = dict(refs)
    print(f"{split}: {len(ids):,} ảnh • {len(annotations):,} captions • {array.shape}")
    del payload, annotations, refs

assert not (set(image_ids["train"]) & set(image_ids["val"]))
assert not (set(image_ids["train"]) & set(image_ids["test"]))
assert not (set(image_ids["val"]) & set(image_ids["test"]))

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

def make_train_loader(seed):
    generator = torch.Generator().manual_seed(seed)
    # num_workers=0 giúp lưu/khôi phục RNG chính xác giữa các epoch.
    loader = DataLoader(caption_datasets["train"], batch_size=BATCH_SIZE,
        shuffle=True, drop_last=False, num_workers=0, pin_memory=True,
        generator=generator)
    return loader, generator

train_loader, train_generator = make_train_loader(44)
val_loader = DataLoader(caption_datasets["val"], batch_size=BATCH_SIZE,
    shuffle=False, drop_last=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(ImageDataset(features["test"], image_ids["test"]),
    batch_size=1, shuffle=False, drop_last=False, num_workers=0, pin_memory=True)
print(f"Vocab={VOCAB_SIZE:,} • GPU={torch.cuda.get_device_name()} • annotations={ANNOTATION_DIR}")
print("Seed được chọn:", SELECTED_SEED)
print("Vùng đầu ra mới (sẽ xác định sau khi kiểm tra protocol):", OUTPUT_ROOT)


Mounted at /content/drive
train: 113,287 ảnh • 566,747 captions • (113287, 512)
val: 5,000 ảnh • 25,010 captions • (5000, 512)
test: 5,000 ảnh • 25,010 captions • (5000, 512)
Vocab=16,214 • GPU=Tesla T4 • annotations=/content/drive/MyDrive/DataKhoaLuan/Data/COCO_2014/annotations_trainval2014/annotations
Seed được chọn: 42
Vùng đầu ra mới (sẽ xác định sau khi kiểm tra protocol): /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/multiseed_beam3


In [3]:
#@title CELL 3 — Model tương thích checkpoint và Beam Search k=3
class BaselineCaptioner(nn.Module):
    def __init__(self, clip_dim=512, vocab_size=VOCAB_SIZE, d_model=256,
                 nhead=8, num_layers=3, dropout=0.1, max_length=30):
        super().__init__()
        self.feature_projection = nn.Linear(clip_dim, d_model)
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(max_length, d_model)
        layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead,
            dim_feedforward=1024, dropout=dropout, batch_first=True,
            activation="relu", norm_first=False)
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_layers)
        self.output_projection = nn.Linear(d_model, vocab_size)
    def forward(self, image_features, input_ids):
        length = input_ids.shape[1]
        if not 1 <= length <= self.position_embedding.num_embeddings:
            raise ValueError(f"Độ dài input không hợp lệ: {length}")
        positions = torch.arange(length, device=input_ids.device).unsqueeze(0)
        # Giữ đúng forward cũ: không nhân token embedding với sqrt(d_model).
        target = self.token_embedding(input_ids) + self.position_embedding(positions)
        memory = self.feature_projection(image_features).unsqueeze(1)
        causal_mask = torch.triu(torch.ones(length, length,
            device=input_ids.device, dtype=torch.bool), diagonal=1)
        hidden = self.decoder(tgt=target, memory=memory, tgt_mask=causal_mask,
                              tgt_key_padding_mask=input_ids.eq(PAD_ID))
        return self.output_projection(hidden)

def new_model():
    return BaselineCaptioner().to(DEVICE)

@torch.inference_mode()
def beam_search_decode(model, image_feature, beam_size=3, max_len=30, alpha=0.6):
    """max_len gồm BOS; mỗi bước xếp hạng cả beam EOS và beam đang mở rộng."""
    if beam_size < 1 or not 2 <= max_len <= model.position_embedding.num_embeddings:
        raise ValueError("beam_size >= 1 và 2 <= max_len <= 30.")
    model.eval()
    device = next(model.parameters()).device
    image_feature = torch.as_tensor(image_feature, dtype=torch.float32, device=device)
    if image_feature.ndim == 2 and image_feature.shape[0] == 1:
        image_feature = image_feature[0]
    if image_feature.shape != (512,):
        raise ValueError(f"Cần một feature 512 chiều, nhận {image_feature.shape}.")
    beams = [([BOS_ID], 0.0)]
    def normalized(beam):
        tokens, score = beam
        return score / (((5.0 + len(tokens) - 1) / 6.0) ** alpha)
    for _ in range(max_len - 1):
        active = [(tokens, score) for tokens, score in beams if tokens[-1] != EOS_ID]
        if not active:
            break
        candidates = [(tokens, score) for tokens, score in beams if tokens[-1] == EOS_ID]
        inputs = torch.tensor([tokens for tokens, _ in active], device=device)
        memory = image_feature.unsqueeze(0).expand(len(active), -1)
        logits = model(memory, inputs)[:, -1, :].float()
        if not torch.isfinite(logits).all():
            raise RuntimeError("Model sinh logits không hữu hạn; dừng để kiểm tra checkpoint.")
        logits[:, [PAD_ID, BOS_ID]] = -torch.inf
        log_prob = torch.log_softmax(logits, dim=-1)
        top_scores, top_ids = log_prob.topk(min(beam_size, VOCAB_SIZE - 2), dim=-1)
        # Một lần chuyển GPU -> CPU cho toàn bộ beam ở bước này.
        for (tokens, score), values, indices in zip(
                active, top_scores.tolist(), top_ids.tolist()):
            candidates.extend((tokens + [int(token)], score + float(value))
                              for value, token in zip(values, indices))
        beams = sorted(candidates, key=normalized, reverse=True)[:beam_size]
        if all(tokens[-1] == EOS_ID for tokens, _ in beams):
            break
    best_tokens, _ = max(beams, key=normalized)
    return " ".join(itos[t] for t in best_tokens if t not in (PAD_ID, BOS_ID, EOS_ID))

ARCHITECTURE = dict(clip_dim=512, d_model=256, nhead=8, num_layers=3,
    dim_feedforward=1024, dropout=0.1, max_length=MAX_LEN, vocab_size=VOCAB_SIZE)
TRAIN_CONFIG = dict(max_epochs=15, patience=3, min_delta=1e-4,
    learning_rate=3e-4, weight_decay=1e-4, batch_size=BATCH_SIZE,
    gradient_clip=1.0, loss_aggregation="caption_weighted_batch_mean_like_original", precision="float32")
DECODE_CONFIG = dict(method="beam_search", beam_size=3, alpha=0.6,
    max_len=MAX_LEN, max_len_includes_bos=True, mask_pad_bos=True,
    implementation="normalized_candidates_v1")
PROTOCOL = dict(architecture=ARCHITECTURE, train=TRAIN_CONFIG,
    decode=DECODE_CONFIG, data=DATA_FINGERPRINTS,
    tokenizer=TOKEN_PATTERN.pattern, evaluator="pycocoevalcap==1.2/PTB/Java11")
EXPECTED_SEED42_SHA256 = "9e9a453975270bb3d5480615cba9a4fed843ff1378b95c7a63b3db4820578dfe"
if sha256_file(SEED42_CHECKPOINT) != EXPECTED_SEED42_SHA256:
    raise ValueError("Checkpoint seed 42 đã đổi so với bản nhóm đã xác minh; dừng trước khi ghi.")
PROTOCOL.update(storage_schema="per_seed_guarded_v2", experiment_tag=EXPERIMENT_TAG,
                seed42_checkpoint_sha256=EXPECTED_SEED42_SHA256)
PROTOCOL_HASH = canonical_hash(PROTOCOL)
METRIC_NAMES = ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4",
                "METEOR", "ROUGE_L", "CIDEr", "SPICE"]

# Mọi đường dẫn ghi đều bị giới hạn vào RUN_DIR; input gốc chỉ được đọc.
import uuid
from contextlib import contextmanager
RUN_DIR = OUTPUT_ROOT / ("run_" + PROTOCOL_HASH[:16])
RESULT_DIR = RUN_DIR
_OBSERVED_FILES = {}
_ACTIVE_LOCKS = {}

def output_path(path):
    path = Path(path).resolve()
    try:
        path.relative_to(RUN_DIR.resolve())
    except ValueError as exc:
        raise PermissionError(f"Từ chối ghi ngoài vùng kết quả mới: {path}") from exc
    return path

def owned_meta(seed, role):
    return {"protocol_hash": PROTOCOL_HASH, "storage_schema": 2,
            "seed": seed, "role": role}

def verify_owner(payload, seed, role):
    expected = owned_meta(seed, role)
    meta = payload.get("_meta", {}) if isinstance(payload, dict) else {}
    if any(meta.get(k) != v for k, v in expected.items()):
        raise ValueError(f"File khác seed/protocol/định dạng ({role}); giữ nguyên và dừng.")

def remember_file(path, before):
    path = output_path(path)
    after = sha256_file(path)
    if before != after:
        raise RuntimeError(f"File đang bị phiên khác thay đổi: {path.name}")
    _OBSERVED_FILES[str(path)] = after

def read_owned_json(path, seed, role):
    path = output_path(path)
    if not path.exists():
        _OBSERVED_FILES[str(path)] = None
        return None
    if not path.is_file():
        raise FileExistsError(f"Đường dẫn đã là thư mục: {path}")
    before = sha256_file(path)
    payload = read_json(path)  # JSON lỗi: dừng, không tự ghi đè.
    verify_owner(payload, seed, role)
    remember_file(path, before)
    return payload

def seed_dir(seed, create=False):
    if seed not in (42, 44, 46):
        raise ValueError("Chỉ có seed 42, 44, 46.")
    path = output_path(RUN_DIR / f"seed_{seed}")
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path

def check_lock(scope):
    lock = _ACTIVE_LOCKS.get(scope)
    if lock is None:
        raise RuntimeError(f"Thiếu khóa ghi cho {scope}.")
    path, token = lock
    try:
        current = read_json(path)
    except (OSError, ValueError) as exc:
        raise RuntimeError("Khóa bị mất hoặc hỏng; dừng ghi.") from exc
    if current.get("token") != token:
        raise RuntimeError("Quyền ghi đã thay đổi; dừng để bảo vệ dữ liệu.")

@contextmanager
def writing_scope(scope):
    global OLD_SESSION_STOPPED
    # Một phiên đang chạy cho mỗi seed. Khóa trên Drive là lớp phòng vệ,
    # không thay thế quy tắc không chạy hai Colab VM cùng seed.
    folder = RUN_DIR if scope == "summary" else seed_dir(scope, create=True)
    lock_path = output_path(folder / ".active_run.json")
    if scope in _ACTIVE_LOCKS:
        raise RuntimeError(f"Seed/phần tổng hợp {scope} đang chạy trong phiên này.")
    if lock_path.exists():
        old_hash = sha256_file(lock_path)
        try:
            old = read_json(lock_path)
        except (OSError, ValueError):
            old = None
        if not OLD_SESSION_STOPPED:
            raise RuntimeError(
                f"Còn phiên/khóa cũ của {scope}. Nếu phiên cũ đã DỪNG HẲN, bật "
                "OLD_SESSION_STOPPED ở Cell 2, rồi chạy lại. Không chạy hai phiên cùng seed.")
        if old is not None:
            verify_owner(old, scope, "active_run")
        elif lock_path.stat().st_size > 4096:
            raise RuntimeError("File khóa hỏng có kích thước bất thường; giữ nguyên để kiểm tra.")
        if sha256_file(lock_path) != old_hash:
            raise RuntimeError("Khóa vừa đổi; không thu hồi.")
        lock_path.unlink()
        OLD_SESSION_STOPPED = False  # Thu hồi một lần, tránh vô tình lấy khóa khác.
    token = uuid.uuid4().hex
    record = {"_meta": owned_meta(scope, "active_run"), "token": token,
              "started_unix": time.time()}
    with lock_path.open("x", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False)
    _ACTIVE_LOCKS[scope] = (lock_path, token)
    try:
        check_lock(scope)
        yield
    finally:
        try:
            if lock_path.is_file() and read_json(lock_path).get("token") == token:
                lock_path.unlink()
        finally:
            _ACTIVE_LOCKS.pop(scope, None)

def required_scope(path):
    relative = output_path(path).relative_to(RUN_DIR.resolve())
    if len(relative.parts) > 1 and relative.parts[0].startswith("seed_"):
        return int(relative.parts[0].split("_")[1])
    return "summary"

def commit_file(path, write_temp, mutable=False):
    path = output_path(path)
    scope = required_scope(path)
    check_lock(scope)
    key = str(path)
    if key not in _OBSERVED_FILES:
        if path.exists():
            raise FileExistsError(f"Chưa xác minh file có sẵn; KHÔNG ghi đè: {path}")
        _OBSERVED_FILES[key] = None
    expected = _OBSERVED_FILES[key]
    current = sha256_file(path) if path.exists() else None
    if current != expected:
        raise RuntimeError(f"File đã đổi sau khi đọc; KHÔNG ghi đè: {path}")
    tmp = output_path(path.with_name("." + path.name + "." + uuid.uuid4().hex + ".tmp"))
    try:
        write_temp(tmp)
        new_hash = sha256_file(tmp)
        if current == new_hash:
            return False
        if current is not None and not mutable:
            raise FileExistsError(f"Kết quả đã có và khác nội dung; giữ nguyên: {path}")
        check_lock(scope)
        if (sha256_file(path) if path.exists() else None) != expected:
            raise RuntimeError(f"File vừa đổi; KHÔNG ghi đè: {path}")
        # Chỉ thay trạng thái đang chạy của đúng experiment/seed đã xác minh.
        # Không di chuyển hay đổi tên bất kỳ dữ liệu/checkpoint nguồn nào.
        os.replace(tmp, path)
        _OBSERVED_FILES[key] = new_hash
        if sha256_file(path) != new_hash:
            raise IOError(f"Readback sau lưu không khớp: {path}")
        return True
    finally:
        # Chỉ xóa file tạm UUID của chính lần ghi này.
        tmp.unlink(missing_ok=True)

def save_json(value, path, mutable=False):
    def write_temp(tmp):
        with tmp.open("x", encoding="utf-8") as f:
            json.dump(value, f, ensure_ascii=False, sort_keys=True,
                      indent=2, allow_nan=False)
            f.flush()
            os.fsync(f.fileno())
    return commit_file(path, write_temp, mutable=mutable)

def initialize_experiment():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = RUN_DIR / "experiment.json"
    manifest = {"_meta": owned_meta(None, "experiment"),
                "protocol": PROTOCOL, "seeds": [42, 44, 46]}
    def validate_manifest():
        last_error = None
        for attempt in range(6):
            try:
                previous = read_owned_json(manifest_path, None, "experiment")
                if previous is not None:
                    if previous != manifest:
                        raise ValueError("Manifest khác cấu hình; không dùng chung thư mục.")
                    return
            except (OSError, json.JSONDecodeError) as exc:
                last_error = exc
            if attempt < 5:
                time.sleep(0.2)
        raise RuntimeError(
            "Manifest thiếu/hỏng hoặc chưa ghi xong; mọi file được giữ nguyên. "
            "Thử chạy lại Cell 3. Nếu vẫn lỗi, đổi EXPERIMENT_TAG ở Cell 2 để "
            "tạo thí nghiệm mới, không xóa/ghi đè thư mục cũ.") from last_error

    if manifest_path.exists():
        validate_manifest()
    elif any(RUN_DIR.iterdir()):
        # Seed khác có thể vừa khởi tạo manifest giữa exists() và iterdir().
        validate_manifest()
    else:
        try:
            with manifest_path.open("x", encoding="utf-8") as f:
                json.dump(manifest, f, ensure_ascii=False, sort_keys=True, indent=2)
                f.flush()
                os.fsync(f.fileno())
        except FileExistsError:
            validate_manifest()
    print("Kết quả mới:", RUN_DIR)
    print(f"Seed đang chọn: {SELECTED_SEED} →", seed_dir(SELECTED_SEED))
    print("Checkpoint/vocab/features/annotations gốc: CHỈ ĐỌC.")

def load_checkpoint(path, expected_seed=None):
    path = Path(path)
    before = sha256_file(path)
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    except Exception as exc:
        raise RuntimeError(f"Không đọc được checkpoint {path}; giữ nguyên file: {exc}") from exc
    if "model_state_dict" not in checkpoint:
        raise ValueError(f"Checkpoint thiếu model_state_dict: {path}")
    if checkpoint.get("stoi") != stoi:
        raise ValueError(f"Vocab trong checkpoint khác vocab.json: {path}")
    cp_itos = checkpoint.get("itos")
    if isinstance(cp_itos, dict):
        cp_itos = [cp_itos[str(i)] for i in range(VOCAB_SIZE)]
    if cp_itos != itos:
        raise ValueError(f"itos trong checkpoint không khớp: {path}")
    if expected_seed is not None:
        output_path(path)
        verify_owner(checkpoint, expected_seed, "checkpoint")
        if checkpoint.get("seed") != expected_seed or checkpoint.get("protocol_hash") != PROTOCOL_HASH:
            raise ValueError(f"Checkpoint khác seed/protocol; giữ nguyên: {path}")
        remember_file(path, before)
    else:
        if checkpoint.get("config", {}).get("seed") != 42:
            raise ValueError("Checkpoint cũ không xác nhận seed 42.")
        if before != EXPECTED_SEED42_SHA256 or sha256_file(path) != before:
            raise ValueError("Checkpoint seed 42 khác hash đã xác minh.")
    return checkpoint

def model_from_checkpoint(path, expected_seed=None):
    checkpoint = load_checkpoint(path, expected_seed)
    model = new_model()
    try:
        model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    except RuntimeError as exc:
        raise RuntimeError(f"Kiến trúc không khớp checkpoint {path}: {exc}") from exc
    return model.eval()

# Kiểm tra cấu trúc nhẹ trước khi thực hiện thí nghiệm.
_model_check = new_model()
assert sum(p.numel() for p in _model_check.parameters()) == 11617110
assert _model_check(torch.zeros(1, 512, device=DEVICE),
                    torch.tensor([[BOS_ID]], device=DEVICE)).shape == (1, 1, VOCAB_SIZE)
del _model_check
torch.cuda.empty_cache()
print("Model tương thích cấu trúc baseline • 11,617,110 tham số • protocol", PROTOCOL_HASH[:12])

initialize_experiment()


Model tương thích cấu trúc baseline • 11,617,110 tham số • protocol fa67e9afd597
Kết quả mới: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/multiseed_beam3/run_fa67e9afd597f3c9
Seed đang chọn: 42 → /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/multiseed_beam3/run_fa67e9afd597f3c9/seed_42
Checkpoint/vocab/features/annotations gốc: CHỈ ĐỌC.


In [4]:
#@title CELL 4 — Hàm đánh giá có resume; chỉ chạy seed 42 khi được chọn
def prediction_prefix_valid(predictions, complete=False):
    if not isinstance(predictions, list) or len(predictions) > len(image_ids["test"]):
        return False
    try:
        ids = [int(row["image_id"]) for row in predictions]
        valid = (ids == image_ids["test"][:len(ids)]
                 and len(ids) == len(set(ids))
                 and all(isinstance(row["caption"], str) for row in predictions))
        return valid and (not complete or len(ids) == 5000)
    except (KeyError, TypeError, ValueError):
        return False

def valid_metrics(metrics):
    return all(name in metrics and isinstance(metrics[name], (int, float))
               and math.isfinite(float(metrics[name])) for name in METRIC_NAMES)

_TOKENIZED_REFERENCES = None
def score_predictions(predictions):
    global _TOKENIZED_REFERENCES
    if not prediction_prefix_valid(predictions, complete=True):
        raise ValueError("Predictions phải phủ đúng 5.000 test IDs theo thứ tự cache.")
    if _TOKENIZED_REFERENCES is None:
        print("PTB tokenize toàn bộ 25.010 reference captions...")
        _TOKENIZED_REFERENCES = PTBTokenizer().tokenize({
            i: [{"caption": caption} for caption in references[i]]
            for i in image_ids["test"]})
    prediction_by_id = {int(p["image_id"]): p["caption"] for p in predictions}
    res = PTBTokenizer().tokenize({
        i: [{"caption": prediction_by_id[i]}] for i in image_ids["test"]})
    metrics = {}
    scorers = [(lambda: Bleu(4), METRIC_NAMES[:4]),
               (ManagedMeteor, ["METEOR"]), (Rouge, ["ROUGE_L"]),
               (Cider, ["CIDEr"]), (Spice, ["SPICE"])]
    for factory, names in scorers:
        print("Đang chấm:", ", ".join(names), flush=True)
        scorer = factory()
        try:
            score, _ = scorer.compute_score(_TOKENIZED_REFERENCES, res)
            for name, value in zip(names, score if len(names) > 1 else [score]):
                value = float(value)
                if not math.isfinite(value):
                    raise ValueError(f"{name} không hữu hạn.")
                metrics[name] = value
        except Exception as exc:
            raise RuntimeError(f"Chấm {names} thất bại; giữ predictions để chấm lại: {exc}") from exc
        finally:
            if isinstance(scorer, ManagedMeteor):
                scorer.close()
            del scorer
            gc.collect()
    if not valid_metrics(metrics):
        raise RuntimeError("Chưa đủ 8 chỉ số hợp lệ; không ghi điểm giả.")
    return metrics

def load_prediction_packet(path, seed, signature):
    packet = read_owned_json(path, seed, "predictions")
    if packet is None:
        return None
    meta = packet["_meta"]
    predictions = packet.get("predictions")
    if meta.get("signature") != signature:
        raise ValueError("Predictions khác checkpoint/cấu hình; GIỮ NGUYÊN, không ghi đè.")
    complete = meta.get("complete")
    if not isinstance(complete, bool) or not prediction_prefix_valid(predictions, complete):
        raise ValueError("Predictions hỏng/trùng/sai thứ tự ID; giữ nguyên và dừng.")
    if meta.get("predictions_hash") != canonical_hash(predictions):
        raise ValueError("Nội dung predictions đã đổi; giữ nguyên và dừng.")
    return packet

def evaluate_seed(seed, checkpoint_path):
    check_lock(seed)
    checkpoint_path = Path(checkpoint_path)
    checkpoint_hash = sha256_file(checkpoint_path)
    checked = load_checkpoint(checkpoint_path, None if seed == 42 else seed)
    if seed != 42 and not checked.get("training_complete", False):
        raise RuntimeError("Checkpoint chưa hoàn tất train; không đánh giá như kết quả cuối.")
    del checked
    metrics_path = seed_dir(seed) / f"metrics_seed{seed}.json"
    predictions_path = seed_dir(seed) / f"predictions_seed{seed}.json"
    signature = canonical_hash({"seed": seed, "checkpoint_sha256": checkpoint_hash,
                               "protocol_hash": PROTOCOL_HASH})
    packet = load_prediction_packet(predictions_path, seed, signature)
    previous = read_owned_json(metrics_path, seed, "metrics")
    if previous is not None:
        if (previous["_meta"].get("signature") != signature or not valid_metrics(previous)
                or previous["_meta"].get("status") != "complete"
                or packet is None or not packet["_meta"]["complete"]
                or previous["_meta"].get("predictions_sha256") != sha256_file(predictions_path)):
            raise ValueError("Có metrics nhưng không xác minh được; GIỮ NGUYÊN và dừng.")
        if sha256_file(checkpoint_path) != checkpoint_hash:
            raise RuntimeError("Checkpoint vừa đổi trong lúc kiểm tra; giữ nguyên kết quả.")
        print(f"Seed {seed}: đã đủ kết quả hợp lệ; bỏ qua, không sửa file.")
        return previous

    predictions = [] if packet is None else list(packet["predictions"])
    start = time.perf_counter()
    def persist_predictions(complete):
        nonlocal packet
        if packet is not None and packet["_meta"]["complete"]:
            raise FileExistsError("Predictions đã hoàn tất; không thay thế.")
        if not prediction_prefix_valid(predictions, complete):
            raise ValueError("Không ghi predictions sai coverage.")
        updated = {
            "_meta": {**owned_meta(seed, "predictions"), "signature": signature,
                "checkpoint_sha256": checkpoint_hash, "complete": complete,
                "num_images": len(predictions),
                "predictions_hash": canonical_hash(predictions),
                "format": "envelope; COCO rows are in the predictions key"},
            "predictions": predictions,
        }
        save_json(updated, predictions_path, mutable=packet is not None)
        # Copy để trạng thái đã lưu không đổi theo list đang append.
        packet = {"_meta": dict(updated["_meta"]), "predictions": list(predictions)}

    if packet is None or not packet["_meta"]["complete"]:
        print(f"Seed {seed}: tiếp tục Beam Search từ ảnh {len(predictions) + 1}/5000.")
        model = model_from_checkpoint(checkpoint_path, None if seed == 42 else seed)
        if sha256_file(checkpoint_path) != checkpoint_hash:
            del model
            raise RuntimeError("Checkpoint đổi trong lúc nạp; dừng.")
        last_saved_count = len(predictions)
        try:
            with tqdm(total=5000, initial=len(predictions),
                      desc=f"Seed {seed} • Beam k=3", unit="ảnh") as progress:
                for index, (ids, batch_features) in enumerate(test_loader):
                    if index < len(predictions):
                        continue
                    assert int(ids[0]) == image_ids["test"][index]
                    caption = beam_search_decode(model, batch_features[0],
                        beam_size=3, max_len=MAX_LEN, alpha=0.6)
                    predictions.append({"image_id": int(ids[0]), "caption": caption})
                    progress.update(1)
                    if len(predictions) % 100 == 0 and len(predictions) < 5000:
                        persist_predictions(complete=False)
                        last_saved_count = len(predictions)
            if sha256_file(checkpoint_path) != checkpoint_hash:
                raise RuntimeError("Checkpoint đã đổi trong khi decode; không chấm điểm lẫn phiên.")
            persist_predictions(complete=True)
        except BaseException:
            if len(predictions) > last_saved_count and len(predictions) < 5000:
                try:
                    persist_predictions(complete=False)
                except Exception as save_error:
                    print("Không lưu thêm được tiến độ; bản đã lưu trước vẫn giữ:", save_error)
            raise
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
    else:
        print(f"Seed {seed}: đã có 5.000 predictions; chỉ chạy phần chấm điểm.")

    if sha256_file(checkpoint_path) != checkpoint_hash:
        raise RuntimeError("Checkpoint đã đổi; không chấm/lưu kết quả cuối.")
    prediction_hash = sha256_file(predictions_path)
    metrics = score_predictions(predictions)
    if (sha256_file(checkpoint_path) != checkpoint_hash
            or sha256_file(predictions_path) != prediction_hash):
        raise RuntimeError("Input đánh giá đã đổi trong lúc chấm; không ghi metrics.")
    metrics["_meta"] = {
        **owned_meta(seed, "metrics"), "status": "complete", "signature": signature,
        "split": "MS COCO 2014 Karpathy Test", "num_images": 5000,
        "num_references": sum(map(len, references.values())),
        "checkpoint": str(checkpoint_path), "checkpoint_sha256": checkpoint_hash,
        "predictions_sha256": prediction_hash, "decoding": DECODE_CONFIG,
        "metric_scale": "raw pycocoevalcap scores, no multiplication by 100",
        "elapsed_seconds_this_session": time.perf_counter() - start,
        "environment": {"python": platform.python_version(), "torch": str(torch.__version__),
            "numpy": np.__version__, "pycocoevalcap": metadata.version("pycocoevalcap"),
            "gpu": torch.cuda.get_device_name(), "java": "11"},
    }
    save_json(metrics, metrics_path)  # Immutable final: không cho thay file khác.
    print(f"Seed {seed} hoàn tất:", {k: round(metrics[k], 6) for k in METRIC_NAMES})
    return metrics

if SELECTED_SEED == 42:
    with writing_scope(42):
        metrics_seed42 = evaluate_seed(42, SEED42_CHECKPOINT)
else:
    print(f"Đang chọn seed {SELECTED_SEED}; Cell 4 chỉ khai báo hàm, không chạy seed 42.")


Seed 42: tiếp tục Beam Search từ ảnh 1/5000.


Seed 42 • Beam k=3:   0%|          | 0/5000 [00:00<?, ?ảnh/s]

PTB tokenize toàn bộ 25.010 reference captions...
Đang chấm: Bleu_1, Bleu_2, Bleu_3, Bleu_4
{'testlen': 47339, 'reflen': 47074, 'guess': [47339, 42339, 37339, 32339], 'correct': [35067, 18998, 9535, 4807]}
ratio: 1.00562943450735
Đang chấm: METEOR
Đang chấm: ROUGE_L
Đang chấm: CIDEr
Đang chấm: SPICE
Seed 42 hoàn tất: {'Bleu_1': 0.740763, 'Bleu_2': 0.576532, 'Bleu_3': 0.439476, 'Bleu_4': 0.335149, 'METEOR': 0.266694, 'ROUGE_L': 0.548353, 'CIDEr': 1.063461, 'SPICE': 0.196479}


In [5]:
#@title CELL 5 — Train mới / chạy tiếp seed 44, 46 rồi đánh giá Beam Search
def save_checkpoint(checkpoint, path):
    verify_owner(checkpoint, checkpoint["seed"], "checkpoint")
    def write_temp(tmp):
        with tmp.open("xb") as f:
            torch.save(checkpoint, f)
            f.flush()
            os.fsync(f.fileno())
    return commit_file(path, write_temp, mutable=True)

def capture_rng(generator):
    np_state = np.random.get_state()
    return {
        "python": random.getstate(), "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
        "numpy": {"name": np_state[0], "keys": np_state[1].tolist(),
                  "pos": int(np_state[2]), "has_gauss": int(np_state[3]),
                  "cached_gaussian": float(np_state[4])},
        "loader": generator.get_state(),
    }

def restore_rng(state, generator):
    random.setstate(state["python"])
    torch.set_rng_state(state["torch_cpu"])
    if torch.cuda.is_available():
        if len(state["torch_cuda"]) != torch.cuda.device_count():
            raise RuntimeError("Số GPU đã đổi; không thể khôi phục RNG chính xác.")
        torch.cuda.set_rng_state_all(state["torch_cuda"])
    n = state["numpy"]
    np.random.set_state((n["name"], np.asarray(n["keys"], dtype=np.uint32),
                         n["pos"], n["has_gauss"], n["cached_gaussian"]))
    generator.set_state(state["loader"])

def run_epoch(model, loader, criterion, epoch, seed, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_items = 0.0, 0
    progress = tqdm(loader, desc=f"Seed {seed} • epoch {epoch:02d} • "
                    + ("train" if training else "val"), leave=False)
    with torch.set_grad_enabled(training):
        for batch_features, tokens in progress:
            batch_features = batch_features.to(DEVICE, non_blocking=True)
            tokens = tokens.to(DEVICE, non_blocking=True)
            input_ids, targets = tokens[:, :-1], tokens[:, 1:]
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(batch_features, input_ids)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss không hữu hạn: seed={seed}, epoch={epoch}")
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
                optimizer.step()
            # Giữ cách tổng hợp epoch loss của notebook baseline seed 42.
            count = int(batch_features.shape[0])
            total_loss += float(loss.detach().item()) * count
            total_items += count
            progress.set_postfix(loss=f"{total_loss / total_items:.4f}")
    if not total_items:
        raise RuntimeError("DataLoader không có caption hợp lệ.")
    return total_loss / total_items

def train_seed(seed):
    seed_everything(seed)
    check_lock(seed)
    if seed not in (44, 46):
        raise ValueError("Không train lại seed 42.")
    best_path = seed_dir(seed) / f"checkpoint_best_seed{seed}.pt"
    last_path = seed_dir(seed) / f"checkpoint_last_seed{seed}.pt"
    for path in (best_path, last_path):
        if path.exists() and not path.is_file():
            raise FileExistsError(f"Đường dẫn checkpoint đã là thư mục; dừng trước train: {path}")
    # Chỉ bỏ qua khi checkpoint xác nhận kết thúc cùng protocol.
    best_existing = load_checkpoint(best_path, seed) if best_path.is_file() else None
    if best_existing is not None and best_existing.get("training_complete", False):
        # Nạp strict trước khi skip để không chấp nhận cấu trúc checkpoint hỏng.
        checked_model = new_model()
        checked_model.load_state_dict(best_existing["model_state_dict"], strict=True)
        del checked_model
        print(f"Seed {seed}: đã train xong, best epoch={best_existing['best_epoch']}; bỏ qua train.")
        del best_existing
        return best_path

    loader, generator = make_train_loader(seed)
    model = new_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    history, best_val_loss, best_epoch, bad_epochs, start_epoch = [], float("inf"), 0, 0, 1
    last_existing = load_checkpoint(last_path, seed) if last_path.is_file() else None
    candidates = [x for x in (last_existing, best_existing) if x is not None]
    if candidates:
        # Nếu ngắt đúng giữa ghi best và last, lấy snapshot epoch mới hơn.
        # Best cũng lưu đủ optimizer/RNG; không coi best tồn tại là đã hoàn tất.
        resume = max(candidates, key=lambda x: int(x["epoch"]))
        required = ("optimizer_state_dict", "rng_state", "history", "best_val_loss",
                    "best_epoch", "epochs_without_improvement")
        if any(key not in resume for key in required):
            raise RuntimeError("Checkpoint chưa đủ trạng thái để resume; không tự bỏ qua/train lại.")
        model.load_state_dict(resume["model_state_dict"], strict=True)
        optimizer.load_state_dict(resume["optimizer_state_dict"])
        history = list(resume["history"])
        best_val_loss = float(resume["best_val_loss"])
        best_epoch = int(resume["best_epoch"])
        bad_epochs = int(resume["epochs_without_improvement"])
        start_epoch = int(resume["epoch"]) + 1
        restore_rng(resume["rng_state"], generator)
        print(f"Seed {seed}: chạy tiếp sau epoch {start_epoch - 1}, best={best_val_loss:.6f}.")
        if not best_path.is_file():
            raise RuntimeError("Có last checkpoint nhưng thiếu best checkpoint; không đánh giá nhầm last.")
    del last_existing, best_existing, candidates
    if "resume" in locals():
        del resume
    gc.collect()

    def snapshot(epoch, complete=False):
        return {
            "_meta": owned_meta(seed, "checkpoint"),
            "epoch": epoch, "seed": seed,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history, "best_val_loss": best_val_loss,
            "best_epoch": best_epoch, "epochs_without_improvement": bad_epochs,
            "training_complete": complete,
            "protocol_hash": PROTOCOL_HASH, "config": PROTOCOL,
            "stoi": stoi, "itos": itos, "rng_state": capture_rng(generator),
        }

    finished_epoch = start_epoch - 1
    try:
        if bad_epochs < TRAIN_CONFIG["patience"]:
            for epoch in range(start_epoch, TRAIN_CONFIG["max_epochs"] + 1):
                train_loss = run_epoch(model, loader, criterion, epoch, seed, optimizer)
                val_loss = run_epoch(model, val_loader, criterion, epoch, seed)
                # Best dựa đúng vào val loss thấp nhất; patience có min_delta=1e-4.
                previous_best = best_val_loss
                improved = val_loss < previous_best
                significant = val_loss < previous_best - TRAIN_CONFIG["min_delta"]
                bad_epochs = 0 if significant else bad_epochs + 1
                if improved:
                    best_val_loss, best_epoch = val_loss, epoch
                history.append({"epoch": epoch, "train_loss": train_loss,
                                "val_loss": val_loss, "bad_epochs": bad_epochs})
                current = snapshot(epoch)
                if improved:
                    save_checkpoint(current, best_path)
                save_checkpoint(current, last_path)
                del current
                finished_epoch = epoch
                print(f"Seed {seed} | {epoch:02d}/15 | train={train_loss:.6f} | "
                      f"val={val_loss:.6f} | best={best_val_loss:.6f} | patience={bad_epochs}/3")
                if bad_epochs >= TRAIN_CONFIG["patience"]:
                    print(f"Seed {seed}: Early Stopping ở epoch {epoch}.")
                    break
        if not best_path.is_file():
            raise RuntimeError(f"Seed {seed} chưa tạo được checkpoint best.")
        # Đánh dấu hoàn thành chỉ sau khi loop kết thúc hợp lệ.
        save_checkpoint(snapshot(finished_epoch, complete=True), last_path)
        completed_best = load_checkpoint(best_path, seed)
        completed_best.update(training_complete=True, finished_epoch=finished_epoch,
            stop_reason="early_stopping" if bad_epochs >= 3 else "max_epochs",
            history=history)
        save_checkpoint(completed_best, best_path)
        del completed_best
    finally:
        del model, optimizer, loader
        gc.collect()
        torch.cuda.empty_cache()
    return best_path

if SELECTED_SEED in (44, 46):
    with writing_scope(SELECTED_SEED):
        checkpoint_path = train_seed(SELECTED_SEED)
        evaluate_seed(SELECTED_SEED, checkpoint_path)
else:
    print("Đang chọn seed 42: Cell 5 không train. Đổi SEED_TO_RUN ở Cell 2 để chạy 44 hoặc 46.")
print("Chạy Cell 6 để xem trạng thái và tổng hợp khi đã đủ 3 seed.")


Đang chọn seed 42: Cell 5 không train. Đổi SEED_TO_RUN ở Cell 2 để chạy 44 hoặc 46.
Chạy Cell 6 để xem trạng thái và tổng hợp khi đã đủ 3 seed.


In [6]:
#@title CELL 6 — Trạng thái từng seed; chỉ tổng hợp khi đủ 3 kết quả cuối
def summarize_completed_seeds():
    ALL_SEEDS = [42, 44, 46]
    seed_metrics = {}
    source_hashes = {}
    for seed in ALL_SEEDS:
        path = seed_dir(seed) / f"metrics_seed{seed}.json"
        prediction_path = seed_dir(seed) / f"predictions_seed{seed}.json"
        if not path.is_file() or not prediction_path.is_file():
            raise FileNotFoundError(f"Thiếu kết quả seed {seed}; hãy chạy Cell 4/5 trước.")
        row = read_owned_json(path, seed, "metrics")
        meta = row.get("_meta", {})
        if not valid_metrics(row) or meta.get("seed") != seed:
            raise ValueError(f"Điểm seed {seed} thiếu hoặc không hợp lệ.")
        if meta.get("protocol_hash") != PROTOCOL_HASH:
            raise ValueError(f"Seed {seed} dùng protocol khác, không được gộp.")
        expected_checkpoint = (SEED42_CHECKPOINT if seed == 42 else
                               seed_dir(seed) / f"checkpoint_best_seed{seed}.pt")
        if Path(meta.get("checkpoint", "")).resolve() != expected_checkpoint.resolve():
            raise ValueError(f"Seed {seed} trỏ nhầm checkpoint.")
        signature = canonical_hash({"seed": seed,
            "checkpoint_sha256": meta["checkpoint_sha256"], "protocol_hash": PROTOCOL_HASH})
        if meta.get("signature") != signature or meta.get("status") != "complete":
            raise ValueError(f"Seed {seed} chưa có metrics cuối đúng signature.")
        packet = load_prediction_packet(prediction_path, seed, signature)
        if packet is None or not packet["_meta"]["complete"]:
            raise ValueError(f"Seed {seed} chưa có đủ 5.000 predictions cuối.")
        if meta.get("predictions_sha256") != sha256_file(prediction_path):
            raise ValueError(f"Predictions seed {seed} đã thay đổi sau khi chấm.")
        checkpoint_path = Path(meta["checkpoint"])
        if sha256_file(checkpoint_path) != meta["checkpoint_sha256"]:
            raise ValueError(f"Checkpoint seed {seed} đã thay đổi sau khi đánh giá.")
        source_hashes[path] = _OBSERVED_FILES[str(output_path(path))]
        source_hashes[prediction_path] = meta["predictions_sha256"]
        source_hashes[checkpoint_path] = meta["checkpoint_sha256"]
        seed_metrics[seed] = row

    statistics = {}
    for name in METRIC_NAMES:
        values = np.asarray([seed_metrics[s][name] for s in ALL_SEEDS], dtype=np.float64)
        statistics[name] = {"mean": float(values.mean()),
                            "std": float(values.std(ddof=1)),
                            "values": {str(s): float(v) for s, v in zip(ALL_SEEDS, values)}}
    summary = {
        "_meta": owned_meta(None, "summary"),
        "metrics_sha256": {str(s): source_hashes[
            seed_dir(s) / f"metrics_seed{s}.json"] for s in ALL_SEEDS},
        "seeds": ALL_SEEDS, "n_seeds": 3, "std_ddof": 1,
        "std_definition": "sample standard deviation across three seeds, not confidence interval",
        "metric_scale": "raw pycocoevalcap scores; multiply ALL values by 100 only if desired",
        "test_images": 5000, "decode": DECODE_CONFIG,
        "protocol_hash": PROTOCOL_HASH, "protocol": PROTOCOL,
        "training_note": "Seed 42 reused (original max_epochs=30); seeds 44/46 max_epochs=15.",
        "comparison_note": "All seeds rescored with Java METEOR; old NLTK METEOR is not interchangeable.",
        "metrics": statistics,
    }
    for path, expected_hash in source_hashes.items():
        if sha256_file(path) != expected_hash:
            raise RuntimeError(f"Input tổng hợp vừa đổi; giữ nguyên summary: {path.name}")
    summary_path = RUN_DIR / "level1_3seeds_beam3_summary.json"
    existing = read_owned_json(summary_path, None, "summary")
    if existing is not None:
        if existing != summary:
            raise ValueError("Summary có sẵn khác nội dung; giữ nguyên, không ghi đè.")
        print("Summary đã có và khớp; không sửa file.")
    else:
        save_json(summary, summary_path)

    labels = {"Bleu_1": "BLEU-1", "Bleu_2": "BLEU-2", "Bleu_3": "BLEU-3",
              "Bleu_4": "BLEU-4", "METEOR": "METEOR", "ROUGE_L": "ROUGE-L",
              "CIDEr": "CIDEr", "SPICE": "SPICE"}
    markdown = [
        "| Chỉ số (thang gốc) | Seed 42 | Seed 44 | Seed 46 | Mean ± Std |",
        "|:--|--:|--:|--:|--:|",
    ]
    for name in METRIC_NAMES:
        st = statistics[name]
        cells = [labels[name]] + [f"{seed_metrics[s][name]:.4f}" for s in ALL_SEEDS]
        cells.append(f"{st['mean']:.4f} ± {st['std']:.4f}")
        markdown.append("| " + " | ".join(cells) + " |")
    display(Markdown("\n".join(markdown)))
    display(Markdown(
        "**Quy ước:** điểm gốc của pycocoevalcap, Std mẫu với ddof=1, n=3. "
        "Đây là biến thiên giữa các seed, không phải khoảng tin cậy. "
        "Seed 42 dùng checkpoint cũ (budget tối đa 30 epoch); seed 44/46 tối đa 15 epoch. "
        "METEOR cả ba seed dùng cùng Java scorer, khác cách tính NLTK cũ."
    ))

    latex = [
        r"\begin{table}[htbp]", r"\centering",
        r"\caption{Baseline Level 1 on COCO 2014 Karpathy Test (5,000 images), "
        r"beam size 3, $\alpha=0.6$. Raw scores; mean $\pm$ sample standard deviation ($n=3$).}",
        r"\label{tab:level1-multiseed-beam3}",
        r"\begin{tabular}{lrrrr}", r"\hline",
        r"Metric & Seed 42 & Seed 44 & Seed 46 & Mean $\pm$ Std \\",
        r"\hline",
    ]
    for name in METRIC_NAMES:
        st = statistics[name]
        cells = [labels[name]] + [f"{seed_metrics[s][name]:.4f}" for s in ALL_SEEDS]
        cells.append("$" + f"{st['mean']:.4f} \\pm {st['std']:.4f}" + "$")
        latex.append(" & ".join(cells) + r" \\")
    latex += [
        r"\hline", r"\end{tabular}", r"\par\smallskip",
        r"{\footnotesize Seed 42: reused checkpoint (original budget 30 epochs); "
        r"seeds 44/46: at most 15 epochs. All use Java METEOR.}",
        r"\end{table}",
    ]
    print("\nLaTeX để dán vào báo cáo:\n")
    print("\n".join(latex))
    print("\nĐã lưu:", summary_path)

missing_seeds = []
for seed in (42, 44, 46):
    folder = seed_dir(seed)
    metric_file = folder / f"metrics_seed{seed}.json"
    pred_file = folder / f"predictions_seed{seed}.json"
    if metric_file.is_file() and pred_file.is_file():
        print(f"Seed {seed}: có kết quả — sẽ xác minh hash/trạng thái trước khi gộp.")
    else:
        missing_seeds.append(seed)
        print(f"Seed {seed}: chưa có đầy đủ kết quả cuối. Thư mục:", folder)

if missing_seeds:
    print("\nChưa tổng hợp; đây là trạng thái bình thường khi chạy riêng từng seed.")
    print("Đổi SEED_TO_RUN ở Cell 2 để chạy các seed còn thiếu:", missing_seeds)
    print("Dữ liệu/điểm cũ không được sửa. Chạy lại Cell 6 sau khi đủ 42, 44, 46.")
else:
    with writing_scope("summary"):
        summarize_completed_seeds()


Seed 42: có kết quả — sẽ xác minh hash/trạng thái trước khi gộp.
Seed 44: chưa có đầy đủ kết quả cuối. Thư mục: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/multiseed_beam3/run_fa67e9afd597f3c9/seed_44
Seed 46: chưa có đầy đủ kết quả cuối. Thư mục: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/multiseed_beam3/run_fa67e9afd597f3c9/seed_46

Chưa tổng hợp; đây là trạng thái bình thường khi chạy riêng từng seed.
Đổi SEED_TO_RUN ở Cell 2 để chạy các seed còn thiếu: [44, 46]
Dữ liệu/điểm cũ không được sửa. Chạy lại Cell 6 sau khi đủ 42, 44, 46.
